In [50]:
################### Sales and Customer Anlytics #######################

# Converting excel to csv since pyspark does not directly support Excel

import pandas as pd
import logging

# Clean up any existing handlers
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# --- Setup Logging ---
log_file = 'sales_analytics.log'
logging.basicConfig(filename=log_file, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Converting Excel to CSV using pandas
try:
  customer_pd_df = pd.read_excel("/content/customer_analytics_dataset.xlsx")
  customer_pd_df.to_csv("/content/customer_data.csv", index=False)
  logging.info("Customer data coverted to CSV successfully")
except Exception as e:
  logging.error(f"Error loading Customer data: {e}")

try:
  sales_pd_df = pd.read_excel("/content/data_analytics_sales_dataset.xlsx")
  sales_pd_df.to_csv("/content/sales_data.csv", index=False)
  logging.info("Sales data coverted to CSV successfully")
except Exception as e:
  logging.error(f"Error loading Customer data: {e}")


!cat sales_analytics.log

2026-10-12 08:14:35,658 - INFO - Customer data coverted to CSV successfully
2026-10-12 08:14:36,760 - INFO - Sales data coverted to CSV successfully


In [51]:
from pyspark.sql import SparkSession
from functools import reduce
from pyspark.sql.functions import col,current_date,to_date

# Creating Spark session
spark = SparkSession.builder.appName("Sales Data Analysis").getOrCreate()


# Loading CSVs into PySpark DataFrames

# Trying to load customer csv file
try:
  customer_df = spark.read.csv("/content/customer_data.csv", header=True, inferSchema=True)
  logging.info("Customer data loaded to spark successfully")
except Exception as e:
  logging.error(f"Error loading Customer data: {e}")


# Trying to load sales csv file
try:
  sales_df = spark.read.csv("/content/sales_data.csv", header=True, inferSchema=True)
  logging.info("Sales data loaded to spark successfully")
except Exception as e:
  logging.error(f"Error loading Sales data: {e}")


# Preview dataframes
customer_df.show(5)
sales_df.show(5)

# --- Show log file contents ---
!cat sales_analytics.log

+-----------+----------------+------------------+---+-------------+--------------+--------------+---------+------------------+----------------+
|Customer_ID|   Customer_Name|              City|Age|Customer_Type|Purchase_Count|Total_Spending| Feedback|Satisfaction_Score|Membership_Years|
+-----------+----------------+------------------+---+-------------+--------------+--------------+---------+------------------+----------------+
|  CUST00001|     Marie Clark|    West Alecmouth| 34|          New|            40|        292758|     Good|                 3|               1|
|  CUST00002|      Donna Hill|North Jamesborough| 54|          New|             5|        173122|     Good|                 3|               5|
|  CUST00003|Jasmine Mcintyre|     Christineberg| 43|      Premium|            15|        425853|     Good|                 9|               7|
|  CUST00004|      Mary Ortiz|       Loriborough| 31|          New|            12|        239889|Excellent|                 7|          

In [52]:
customer_df.count()

8000

In [53]:
sales_df.count()

8000

In [54]:
################# Data Cleaning and Transformation#####################

# Remove duplicates
customer_df = customer_df.dropDuplicates()
sales_df = sales_df.dropDuplicates()
logging.info("Duplicates removed successfully")

# Correct data types (e.g., ensuring Order date is in a Date format, Quantity is an integer, etc.).
customer_df.printSchema()
customer_df.show(5)

sales_df.printSchema()
sales_df.show(5)

logging.info("Data types corrected successfully")

root
 |-- Customer_ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Customer_Type: string (nullable = true)
 |-- Purchase_Count: integer (nullable = true)
 |-- Total_Spending: integer (nullable = true)
 |-- Feedback: string (nullable = true)
 |-- Satisfaction_Score: integer (nullable = true)
 |-- Membership_Years: integer (nullable = true)

+-----------+---------------+---------------+---+-------------+--------------+--------------+--------+------------------+----------------+
|Customer_ID|  Customer_Name|           City|Age|Customer_Type|Purchase_Count|Total_Spending|Feedback|Satisfaction_Score|Membership_Years|
+-----------+---------------+---------------+---+-------------+--------------+--------------+--------+------------------+----------------+
|  CUST00137|   Jacob Dorsey|Christopherland| 31|          New|            43|        414851|    Good|                10|               3|

In [55]:
# Handle missing or null values by either dropping or filling them.

# Identifying the null values
customer_df_with_missing_values = customer_df.filter(
    reduce(lambda a, b: a | b, [col(c).isNull() for c in customer_df.columns])
)
customer_df_with_missing_values.show()

sales_df_with_missing_values = sales_df.filter(
    reduce(lambda a, b: a | b, [col(c).isNull() for c in sales_df.columns])
)
sales_df_with_missing_values.show()

logging.info("Null values identified successfully")

# Dropping the null rows

customer_df=customer_df.dropna()
sales_df=sales_df.dropna()

logging.info("Null values removed successfully")

+-----------+-------------+----+---+-------------+--------------+--------------+--------+------------------+----------------+
|Customer_ID|Customer_Name|City|Age|Customer_Type|Purchase_Count|Total_Spending|Feedback|Satisfaction_Score|Membership_Years|
+-----------+-------------+----+---+-------------+--------------+--------------+--------+------------------+----------------+
+-----------+-------------+----+---+-------------+--------------+--------------+--------+------------------+----------------+

+--------+----+-------------+------+-------+--------+----------+-----+-------+------+------------+---------------+
|Order_ID|Date|Customer_Name|Region|Product|Quantity|Unit_Price|Sales|Expense|Profit|Payment_Mode|Customer_Rating|
+--------+----+-------------+------+-------+--------+----------+-----+-------+------+------------+---------------+
+--------+----+-------------+------+-------+--------+----------+-----+-------+------+------------+---------------+



In [56]:
# Renaming the column names to snake-case for consistency

sales_df = sales_df.withColumnRenamed("date", "order_date")

customer_df = customer_df.select([col(c).alias(c.lower().replace(' ', '_')) for c in customer_df.columns])
customer_df.show(5)
sales_df = sales_df.select([col(c).alias(c.lower().replace(' ', '_')) for c in sales_df.columns])


logging.info("Column names renamed successfully")

# Adding New Column Month_Year
from pyspark.sql.functions import date_format

sales_df = sales_df.withColumn(
    "year_month",
    date_format("order_date", "yyyy-MM")
)

sales_df.show(5)

+-----------+---------------+---------------+---+-------------+--------------+--------------+--------+------------------+----------------+
|customer_id|  customer_name|           city|age|customer_type|purchase_count|total_spending|feedback|satisfaction_score|membership_years|
+-----------+---------------+---------------+---+-------------+--------------+--------------+--------+------------------+----------------+
|  CUST00137|   Jacob Dorsey|Christopherland| 31|          New|            43|        414851|    Good|                10|               3|
|  CUST00140|Stephanie Smith| Kaitlynchester| 41|      Premium|            34|        484825|    Poor|                 7|               2|
|  CUST00195| Nicholas Irwin|       Jillfort| 39|    Returning|            44|        165147| Average|                 5|               1|
|  CUST00245|   Ronald Adams|    South Joyce| 20|    Returning|            31|        496548| Average|                 4|              10|
|  CUST00678|  Scott Johnso

In [57]:
# Filling the missing values with dafault

customer_df= customer_df.fillna({
    "age":0,
    "city":"unknown",
    "purchase_count":0,
    "satisfaction_score":0
})

sales_df=sales_df.fillna({
    "region":"unknown",
    "sales":0,
    "quantity":0,
    "profit":0
})

logging.info("Missing values filled successfully")

In [58]:
# Creating a temporary view
customer_df.createOrReplaceTempView("customer_data")
sales_df.createOrReplaceTempView("sales_data")
logging.info("Temporary views created successfully")
print("Temporary views created successfully")
customer_df=spark.sql("select * from customer_data")
sales_df=spark.sql("select * from  sales_data order by year_month desc")

customer_df.show(5)
customer_df.count()
sales_df.show(5)
sales_df.count()

Temporary views created successfully
+-----------+---------------+---------------+---+-------------+--------------+--------------+--------+------------------+----------------+
|customer_id|  customer_name|           city|age|customer_type|purchase_count|total_spending|feedback|satisfaction_score|membership_years|
+-----------+---------------+---------------+---+-------------+--------------+--------------+--------+------------------+----------------+
|  CUST00137|   Jacob Dorsey|Christopherland| 31|          New|            43|        414851|    Good|                10|               3|
|  CUST00140|Stephanie Smith| Kaitlynchester| 41|      Premium|            34|        484825|    Poor|                 7|               2|
|  CUST00195| Nicholas Irwin|       Jillfort| 39|    Returning|            44|        165147| Average|                 5|               1|
|  CUST00245|   Ronald Adams|    South Joyce| 20|    Returning|            31|        496548| Average|                 4|        

8000

In [59]:
######## Level 1 ########
######## Task 1 #########

############ Sales Performance Analysis #############

# Calculate total sales,profit,orders

total_sales_df = spark.sql("""
select concat(round(sum(sales)/10000000,2),'Cr') as total_sales,
concat(round(sum(expense)/10000000,2),'Cr') as total_expense,
concat(round(sum(profit)/10000000,2),'Cr') as total_profit,
round(sum(profit)/sum(sales)*100,2) as profit_margin,
count(distinct order_id) as total_orders
from sales_data
""")

total_sales_df.show()

# Identify Top 5 Selling Products

top5_products_df = spark.sql("""
select product, sum(quantity) as total_quantity
from sales_data
group by product
order by total_quantity desc
limit 5
""")

top5_products_df.show()

# Compare region wise sales

region_sales_df =spark.sql("""
select region, sum(sales) as total_sales
from sales_data
group by region
order by total_sales desc
""")

region_sales_df.show()

############### Customer Feedback Analysis #######################

########## Task 2 of Level 1 #################


# Count positive and negative feedback

feedback_df=spark.sql("""
select
  distinct feedback,
  count(*) as count
from customer_data
group by feedback
""")
feedback_df.show()


# Calculate average satisfaction score

satisfaction_score_df=spark.sql("""
select round(avg(satisfaction_score),2) as avg_satisfaction_score
from customer_data
""")
satisfaction_score_df.show()


# Ananalyze premium customers

premium_customers_df=spark.sql("""
select customer_name,total_spending
from customer_data
order by total_spending desc
limit 10
""")
premium_customers_df.show()

########## Task 3 of Level 1 #################

# Top Spending customers

top_spending_customer_df=spark.sql("""
select customer_name,
customer_type,
total_spending
from customer_data
order by total_spending desc
limit 10
""")
top_spending_customer_df.show()

# Average spending per customer

avg_spending_df=spark.sql("""
select round(avg(total_spending),2) as avg_spending
from customer_data
""")
avg_spending_df.show()

# Age group spending

age_group_spending_df=spark.sql("""
select
  case
    when age between 18 and 30 then '18-30'
    when age between 31 and 40 then '31-40'
    when age between 41 and 50 then '41-50'
    else '50+'
  end as age_group,
  count(*) as count,
  sum(total_spending) as total_spending,
  round(avg(total_spending),2) as avg_spending
from customer_data
group by case
    when age between 18 and 30 then '18-30'
    when age between 31 and 40 then '31-40'
    when age between 41 and 50 then '41-50'
    else '50+' end
order by age_group
""")
age_group_spending_df.show()




+-----------+-------------+------------+-------------+------------+
|total_sales|total_expense|total_profit|profit_margin|total_orders|
+-----------+-------------+------------+-------------+------------+
|   201.78Cr|     141.63Cr|     60.15Cr|        29.81|        8000|
+-----------+-------------+------------+-------------+------------+

+----------+--------------+
|   product|total_quantity|
+----------+--------------+
|Headphones|          9242|
|    Tablet|          8961|
|Smartwatch|          8893|
|    Laptop|          8689|
|    Mobile|          8064|
+----------+--------------+

+------+-----------+
|region|total_sales|
+------+-----------+
|  West|  514715058|
|  East|  503852183|
| South|  501368855|
| North|  497835412|
+------+-----------+

+---------+-----+
| feedback|count|
+---------+-----+
|Excellent| 1983|
|  Average| 1955|
|     Good| 2064|
|     Poor| 1998|
+---------+-----+

+----------------------+
|avg_satisfaction_score|
+----------------------+
|                

In [60]:
#############  Level 3 ###############

# Predictive sales forecasting

monthly_sales_df = spark.sql("""
select
  date_format(order_date, 'yyyy-MM') as month,
  sum(sales) as total_sales
from sales_data
group by date_format(order_date, 'yyyy-MM')
order by month
""")

monthly_sales_df.show()

monthly_sales_df.show(50, truncate=False)


# Adding month number
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.orderBy("month")

monthly_sales_numbered = monthly_sales_df.withColumn(
    "month_number",
    row_number().over(window)
)

monthly_sales_numbered.show(20)

# ML Model

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

assembler = VectorAssembler(
    inputCols=["month_number"],
    outputCol="features"
)

forecast_data = assembler.transform(monthly_sales_numbered)

lr = LinearRegression(
    featuresCol="features",
    labelCol="total_sales"
)

model = lr.fit(forecast_data)

print("Slope:", model.coefficients[0])
print("Intercept:", model.intercept)


# Prediction

predictions = model.transform(forecast_data)

predictions.select(
    "month",
    "total_sales",
    "prediction"
).show(20)


# Forecasing next 6 months
future_months = spark.createDataFrame([
    ("2025-06", 18),
    ("2025-07", 19),
    ("2025-08", 20),
    ("2025-09", 21),
    ("2025-10", 22),
    ("2025-11", 23)
], ["month", "month_number"])

future_data = assembler.transform(future_months)
future_predictions = model.transform(future_data)

future_predictions.select(
    "month",
    "month_number",
    "prediction"
).show()


# Actual Vs Forecast

historical = monthly_sales_numbered.select(
    "month",
    "month_number",
    "total_sales"
).withColumnRenamed(
    "total_sales",
    "actual_sales"
)

forecast = future_predictions.select(
    "month",
    "month_number",
    "prediction"
).withColumnRenamed(
    "prediction",
    "forecast_sales"
)

actual_vs_forecast = historical.unionByName(
    forecast,
    allowMissingColumns=True
).orderBy("month_number")



# Remove month_number
final_forecast = actual_vs_forecast.drop("month_number")

final_forecast.show(30, truncate=False)


+-------+-----------+
|  month|total_sales|
+-------+-----------+
|2024-01|  131139380|
|2024-02|  121362934|
|2024-03|  120130920|
|2024-04|  109018924|
|2024-05|  125222899|
|2024-06|  122627143|
|2024-07|  116592901|
|2024-08|  121246944|
|2024-09|  117191345|
|2024-10|  124832410|
|2024-11|  129692840|
|2024-12|  138104559|
|2025-01|  119305964|
|2025-02|  103756068|
|2025-03|  127228945|
|2025-04|  132153799|
|2025-05|   58163533|
+-------+-----------+

+-------+-----------+
|month  |total_sales|
+-------+-----------+
|2024-01|131139380  |
|2024-02|121362934  |
|2024-03|120130920  |
|2024-04|109018924  |
|2024-05|125222899  |
|2024-06|122627143  |
|2024-07|116592901  |
|2024-08|121246944  |
|2024-09|117191345  |
|2024-10|124832410  |
|2024-11|129692840  |
|2024-12|138104559  |
|2025-01|119305964  |
|2025-02|103756068  |
|2025-03|127228945  |
|2025-04|132153799  |
|2025-05|58163533   |
+-------+-----------+

+-------+-----------+------------+
|  month|total_sales|month_number|
+---

In [61]:
# Converting the sql results to Pandas

# Level 1

total_sales_pd = total_sales_df.toPandas()
top5_products_pd = top5_products_df.toPandas()
region_sales_pd = region_sales_df.toPandas()
feedback_df_pd = feedback_df.toPandas()
satisfaction_score_df_pd = satisfaction_score_df.toPandas()
premium_customers_df_pd = premium_customers_df.toPandas()
top_spending_customer_pd = top_spending_customer_df.toPandas()
avg_spending_df_pd = avg_spending_df.toPandas()
age_group_spending_pd = age_group_spending_df.toPandas()

# Level 2
customer_pd=customer_df.toPandas()
sales_pd=sales_df.toPandas()

# Level 3

import numpy as np
import pandas as pd

final_forecast_pd = final_forecast.toPandas()





In [62]:
# Installing the required library for using google sheet

!pip install -q gspread

In [63]:
# Authenticating the google account

from google.colab import auth
auth.authenticate_user()

In [64]:
# Connecting to google sheet

import gspread
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

In [65]:
# Creating Google sheet

#spreadsheet = gc.create("Level1 Analysis")
#print("Google Sheet created:", spreadsheet.url)



#spreadsheet = gc.create("Level2 Analysis")
##print("Google Sheet created:", spreadsheet.url)



#spreadsheet = gc.create("Level3 Analysis")
#print("Google Sheet created:", spreadsheet.url)

In [66]:
# Write Level 1 results to Google Sheets


import gspread

# Open the Google Spreadsheet
spreadsheet = gc.open("Level1 Analysis")


# Function to write/update a worksheet
def write_to_sheet(spreadsheet, dataframe, sheet_name):

    # Try to open existing worksheet
    try:
        worksheet = spreadsheet.worksheet(sheet_name)

        # Clear old data
        worksheet.clear()

    except gspread.WorksheetNotFound:

        # Create worksheet if it doesn't exist
        worksheet = spreadsheet.add_worksheet(
            title=sheet_name,
            rows=100,
            cols=10
        )

    # Convert DataFrame to list
    data = [dataframe.columns.tolist()] + dataframe.values.tolist()

    # Write data
    worksheet.update(
        range_name="A1",
        values=data
    )



# Writing all Level 1 tables


write_to_sheet(
    spreadsheet,
    total_sales_pd,
    "Total Sales"
)

write_to_sheet(
    spreadsheet,
    top5_products_pd,
    "Top5 Products"
)

write_to_sheet(
    spreadsheet,
    region_sales_pd,
    "Region Sales"
)

write_to_sheet(
    spreadsheet,
    feedback_df_pd,
    "Feedback Analysis"
)

write_to_sheet(
    spreadsheet,
    satisfaction_score_df_pd,
    "Average Satisfaction Score"
)

write_to_sheet(
    spreadsheet,
    premium_customers_df_pd,
    "Premium Customers"
)

write_to_sheet(
    spreadsheet,
    top_spending_customer_pd,
    "Top Spending Customers"
)

write_to_sheet(
    spreadsheet,
    avg_spending_df_pd,
    "Average Spending"
)

write_to_sheet(
    spreadsheet,
    age_group_spending_pd,
    "Age Group Spending"
)


print("Google Sheets updated successfully!")
print("Level 1 completed successfully")


logging.info("Level 1 completed successfully")

Google Sheets updated successfully!
Level 1 completed successfully


In [67]:
# Write Level 2 results to Google Sheets

from datetime import date, datetime
import gspread

# Open the Google Spreadsheet
spreadsheet = gc.open("Level2 Analysis")


# Function to write/update a worksheet
def write_to_sheet(spreadsheet, dataframe, sheet_name):

    # Try to open existing worksheet
    try:
        worksheet = spreadsheet.worksheet(sheet_name)

        # Clear old data
        worksheet.clear()

    except gspread.WorksheetNotFound:

        # Create worksheet if it doesn't exist
        worksheet = spreadsheet.add_worksheet(
            title=sheet_name,
            rows=9000,
            cols=20
        )
    # Convert date and datetime columns to strings
    dataframe = dataframe.copy()

    for col in dataframe.columns:
        dataframe[col] = dataframe[col].apply(
            lambda x: x.isoformat()
            if isinstance(x, (date, datetime, pd.Timestamp))
            else x
        )

    # Convert DataFrame to list
    data = [dataframe.columns.tolist()] + dataframe.values.tolist()

    # Write data
    worksheet.update(
        range_name="A1",
        values=data
    )



# Writing all Level 2 tables

write_to_sheet(
    spreadsheet,
    customer_pd,
    "Customer"
)

write_to_sheet(
    spreadsheet,
    sales_pd,
    "Sales"
)


print("Google Sheets updated successfully!")
print("Level 2 completed successfully")


logging.info("Level 2 completed successfully")

Google Sheets updated successfully!
Level 2 completed successfully


In [68]:
# Write Level 3  results to Google Sheets

from datetime import date, datetime
import gspread

# Open the Google Spreadsheet
spreadsheet = gc.open("Level3 Analysis")


# Function to write/update a worksheet
def write_to_sheet(spreadsheet, dataframe, sheet_name):

    # Try to open existing worksheet
    try:
        worksheet = spreadsheet.worksheet(sheet_name)

        # Clear old data
        worksheet.clear()

    except gspread.WorksheetNotFound:

        # Create worksheet if it doesn't exist
        worksheet = spreadsheet.add_worksheet(
            title=sheet_name,
            rows=9000,
            cols=20
        )
    # Convert date and datetime columns to strings
    dataframe = dataframe.copy()

    for col in dataframe.columns:
        dataframe[col] = dataframe[col].apply(
            lambda x: x.isoformat()
            if isinstance(x, (date, datetime, pd.Timestamp))
            else x
        )

    dataframe = dataframe.replace([np.inf, -np.inf], np.nan)

    dataframe = dataframe.astype(object).where(
    pd.notna(dataframe), None
    )

    # Convert DataFrame to list
    data = [dataframe.columns.tolist()] + dataframe.values.tolist()

    # Write data
    worksheet.update(
        range_name="A1",
        values=data
    )



# Writing all Level 3 tables

write_to_sheet(
    spreadsheet,
    customer_pd,
    "Customer"
)

write_to_sheet(
    spreadsheet,
    sales_pd,
    "Sales"
)

write_to_sheet(
    spreadsheet,
    final_forecast_pd,
    "Actual Vs Forecast"
)


print("Google Sheets updated successfully!")
print("Level 3 completed successfully")


logging.info("Level 3 completed successfully")

Google Sheets updated successfully!
Level 3 completed successfully
